## ⏮️ Funciones de Ventana: LAG y LEAD en SQL

**Explicación breve:**
Las funciones **LAG** y **LEAD** permiten comparar filas dentro de una tabla sin necesidad de hacer subconsultas.

Se usan para mirar datos **anteriores** o **siguientes** dentro de un orden específico.

### 🔹 LAG

Permite ver un valor de una **fila anterior**.

👉 Sirve para comparar el presente con el pasado.
Ejemplo típico: comparar la nota actual de un estudiante con su nota anterior.

### 🔹 LEAD

Permite ver un valor de una **fila siguiente**.

👉 Sirve para anticipar valores futuros dentro del conjunto de datos.
Ejemplo típico: ver la siguiente calificación después de un examen.

---

### 🧠 En resumen fácil

| Función | Mira hacia | Uso típico                    |
| ------- | ---------- | ----------------------------- |
| LAG     | ⬅️ Pasado  | Comparar con datos anteriores |
| LEAD    | ➡️ Futuro  | Comparar con datos siguientes |

---



In [2]:
import duckdb
import os
os.listdir('../data/student_performance')
con = duckdb.connect()
con.execute("CREATE TABLE student_performance AS SELECT * FROM read_csv_auto('../data/student_performance/student_performance.csv')")
result = con.execute("SELECT * FROM student_performance").fetchdf()
print(result.head())


   student_id  Hours_Studied  Attendance Parental_Involvement  \
0           1             23          84                  Low   
1           2             19          64                  Low   
2           3             24          98               Medium   
3           4             29          89                  Low   
4           5             19          92               Medium   

  Access_to_Resources  Extracurricular_Activities  Sleep_Hours  \
0                High                       False            7   
1              Medium                       False            8   
2              Medium                        True            7   
3              Medium                        True            8   
4              Medium                        True            6   

   Previous_Scores Motivation_Level  Internet_Access  ...  Family_Income  \
0               73              Low             True  ...            Low   
1               59              Low             True  ...   

# LAG #1 — Rendimiento marginal por horas de estudio:


In [3]:
con.execute("""
 SELECT
    hours_studied,
    exam_score,
    LAG(exam_score) OVER (ORDER BY hours_studied) AS score_estudiante_anterior,
    exam_score - LAG(exam_score) OVER (ORDER BY hours_studied) AS diferencia_marginal
FROM student_performance
ORDER BY hours_studied;           
            """).fetch_df()

,Hours_Studied,Exam_Score,score_estudiante_anterior,diferencia_marginal
0,1,92,<NA>,<NA>
1,1,61,92,-31
2,1,60,61,-1
3,2,66,60,6
4,2,65,66,-1
...,...,...,...,...
6602,39,75,73,2
6603,39,77,75,2
6604,39,78,77,1
6605,43,78,78,0


# LAG #2 — Sueño y rendimiento:


In [4]:
con.execute("""
SELECT
    exam_score,
    sleep_hours,
    LAG(sleep_hours) OVER (ORDER BY exam_score DESC) AS sleep_estudiante_superior,
    sleep_hours - LAG(sleep_hours) OVER (ORDER BY exam_score DESC) AS diff_sueno
FROM student_performance
ORDER BY exam_score DESC;
            """).fetch_df()

,Exam_Score,Sleep_Hours,sleep_estudiante_superior,diff_sueno
0,101,6,<NA>,<NA>
1,100,4,6,-2
2,99,8,4,4
3,99,4,8,-4
4,98,9,4,5
...,...,...,...,...
6602,57,8,6,2
6603,57,7,8,-1
6604,57,10,9,1
6605,56,7,7,0


# LEAD #1 — Brecha al siguiente rango:


In [5]:
con.execute("""
SELECT
    ROW_NUMBER() OVER (ORDER BY exam_score) AS student_id,
    exam_score,
    LEAD(exam_score) OVER (ORDER BY exam_score) AS score_siguiente,
    LEAD(exam_score) OVER (ORDER BY exam_score) - exam_score AS puntos_para_subir
FROM student_performance
ORDER BY exam_score;
""").fetch_df()

,student_id,Exam_Score,score_siguiente,puntos_para_subir
0,1,55,56,1
1,2,56,57,1
2,3,57,57,0
3,4,57,57,0
4,5,57,57,0
...,...,...,...,...
6602,6603,98,99,1
6603,6604,99,99,0
6604,6605,99,100,1
6605,6606,100,101,1


# LEAD #2 — Proyección de tutorías:


In [6]:
con.execute("""
SELECT
    ROW_NUMBER() OVER (ORDER BY tutoring_sessions, exam_score) AS student_id,
    tutoring_sessions,
    exam_score,
    LEAD(exam_score) OVER (ORDER BY tutoring_sessions, exam_score) AS score_con_mas_tutorias,
    LEAD(exam_score) OVER (ORDER BY tutoring_sessions, exam_score) - exam_score AS salto_potencial
FROM student_performance
ORDER BY tutoring_sessions, exam_score;
""").fetch_df()

,student_id,Tutoring_Sessions,Exam_Score,score_con_mas_tutorias,salto_potencial
0,1,0,56,57,1
1,2,0,57,57,0
2,3,0,57,57,0
3,4,0,57,58,1
4,5,0,58,58,0
...,...,...,...,...,...
6602,6603,7,71,71,0
6603,6604,7,71,73,2
6604,6605,7,73,75,2
6605,6606,7,75,69,-6
